# Full pipeline walkthrough

This notebook walks through **every stage** of the TomoImageStitcher
pipeline on a tiny **synthetic dataset** so you can run it end-to-end on
a laptop GPU in under a minute.

It is the **canonical tutorial** for the package. The README gives you a
10-line overview; this notebook gives you the per-stage explanation, the
parameter choices, and the visualisations.

If you are here to stitch real data, read the [installation guide]
(../docs/installation.md) first, then come back to this notebook and
swap the synthetic data for your own `.h5` files.

**Prerequisites**

- `tomo-image-stitcher[notebook]` installed
- `cupy-cuda12x` (or matching your CUDA version) installed
- A CUDA-capable GPU

Mathematical details (ZNCC, IC-GN Lucas–Kanade, mask weighting,
distance-map blending) live in [PUBLICATION].


In [ ]:
import os
import shutil
import numpy as np
import h5py
import matplotlib.pyplot as plt

from tomo_image_stitcher import Stitcher

np.random.seed(0)


## 0. Generate a tiny synthetic dataset

We build **three overlapping 3D sub-volumes** on disk. Each sub-volume is
a 40 × 40 × 40 cube that contains a small bright feature (a synthetic
"sample") and lots of background (zeros).

The three sub-volumes are shifted relative to one another along `x`,
which simulates a translation-stage acquisition. The known shifts are
stored in `motor_coords` and serve as a ground truth we can compare to.

**Why synthetic data?** It runs in a few seconds on any GPU, and we know
the answer in advance, so we can sanity-check every stage.


In [ ]:
WORK = "tutorial_workspace"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
os.makedirs(WORK)

n_subvols = 3
shape = (40, 40, 40)         # (Z, Y, X) per sub-volume
background = 0.001           # value for empty voxels
feature_value = 0.005        # value for the bright spot

# 8 pixels of overlap on x, then a 16-pixel gap to the next sub-volume
# Sub-volume centres:  0,  24,  48  (in voxels) -> 0, 24, 48 mm at 1 mm/vox
centres_vox = np.array([0, 24, 48])
mm_per_vox = 1.0
centres_mm = centres_vox * mm_per_vox

file_paths = []
for i, cx in enumerate(centres_vox):
    vol = np.full(shape, background, dtype=np.float32)
    # add a small bright cube in the centre of each sub-volume
    vol[18:22, 18:22, 18:22] = feature_value
    # add a per-volume intensity offset (simulates detector drift)
    vol = vol + 0.0005 * i
    path = os.path.join(WORK, f"scan_{i:03d}.h5")
    with h5py.File(path, "w") as f:
        f.create_dataset("data", data=vol)
    file_paths.append(path)

print("Created sub-volumes at centres (mm):", centres_mm.tolist())
print("Files:", file_paths)


In [ ]:
fig, axes = plt.subplots(1, n_subvols, figsize=(3 * n_subvols, 3))
for i, path in enumerate(file_paths):
    with h5py.File(path, "r") as f:
        vol = f["data"][:]
    mid = vol.shape[0] // 2
    axes[i].imshow(vol[mid], cmap="gray")
    axes[i].set_title(f"scan {i}: mid-Z slice")
    axes[i].axis("off")
plt.tight_layout()
plt.show()


## 1. Organise sub-volumes

The pipeline starts with a **list of `.h5` files** and a matching list
of **motor coordinates in millimetres**. The motor positions come from
the beamline acquisition; each sub-volume is a `.h5` reconstruction,
and the motor positions tell us approximately where each one sits.

In this stage the Stitcher:

1. Reads only the slices it needs (no full-volume load).
2. **Classifies** sub-volumes into **z-layers** (sub-volumes at the same
   height are processed together).
3. Computes the **global padding box** — the size of the final canvas.
4. Finds the **intersections** between every pair of neighbours — these
   small overlapping 3D blocks are the units of registration.

Math reference: [PUBLICATION].


In [ ]:
st = Stitcher(
    file_path_list=file_paths,
    physical_coordinates=centres_mm,
    mm_per_voxel=mm_per_vox,
    x_y_z_correspondance=(1, 2, 3),   # (x, y, z) axes in our data
    saving_path=os.path.join(WORK, "pipeline_meta"),
)
print("Stitcher ready:", st)


### 1.1. Classify into z-layers

Sub-volumes at the same height belong to the same layer. We use a
`tolerance_mm` that is larger than the maximum motor jitter but smaller
than the spacing between distinct acquisition heights.

In our synthetic data there is only one height, so all sub-volumes end
up in layer 0.


In [ ]:
layers = st.get_layers_in_z(tolerance_mm=2.0)
print("Layer count:", len(layers))
print("Layer 0 sub-volume indexes:", layers[0])


### 1.2. Compute the global padding box

The padding box is the bounding box in the global frame that contains
all sub-volumes after we place them at their motor coordinates. The
final stitched volume will be exactly this size.


In [ ]:
st.get_padding()
print("Padding (z, y, x):", st.img_depth, st.img_height, st.img_width)


We can preview the layout with `check_padding`. With `circular_mask=True`
it overlays a circular mask on the centre of each sub-volume so you can
spot which sub-volumes overlap which.


In [ ]:
preview = st.check_padding(layer_index=0, circular_mask=True)
plt.figure(figsize=(5, 5))
plt.imshow(preview[preview.shape[0] // 2], cmap="gray")
plt.title("Global padding preview (mid-Z)")
plt.axis("off")
plt.show()


### 1.3. Find the intersections

An **intersection** is the 3D block that two neighbouring sub-volumes
have in common. The Stitcher records one intersection per neighbour
pair, parameterised by the index of the two sub-volumes and the slice
ranges they share.

`check=True` prints a short summary of how many intersections were
found and on which layers.


In [ ]:
st.get_intersections(check=True)
print("Layer 0 intersection count:", len(st.intersections[0]))


`check_intersection` extracts one intersection as a numpy array so we
can inspect it. The result below is the overlap of sub-volumes 0 and 1
as seen in the middle slice.


In [ ]:
inter = st.check_intersection(layer=0, image=0, intersection=0,
                              mask=False, mask_radius=None)
print("Intersection shape (Z, Y, X):", inter.shape)

plt.figure(figsize=(4, 4))
plt.imshow(inter[inter.shape[0] // 2], cmap="gray")
plt.title("Intersection 0 (mid-Z)")
plt.axis("off")
plt.show()


## 2. Registration

Registration finds the **sub-pixel 3D shift** between two neighbouring
sub-volumes. It runs once per neighbour pair and produces, for each
pair, a translation (or affine) plus a quality score (the NCC, in
`[-1, 1]`, where 1 is a perfect match).

The TomoImageStitcher registration has two engines:

1. **ZNCC pixel search.** Zero-mean Normalised Cross-Correlation in
   Fourier space, on the GPU. Robust to intensity offsets and to global
   scaling. Runs coarse-to-fine across multiple downscaling stages so a
   large shift can be found in O(log N) time.
2. **IC-GN Lucas–Kanade refinement.** An inverse-compositional
   Gauss–Newton optimiser that brings the coarse shift to **sub-pixel**
   accuracy. The warp can be `translation`, `rigid`, or `affine`.

Math reference: [PUBLICATION].

### Choosing parameters

- `downscale`: factor for the coarsest ZNCC stage. Higher = faster, less
  accurate. Good values are 2–8.
- `downscale_stages`: number of ZNCC pyramid levels. More = handles
  larger shifts. 3–5 is typical.
- `mask=True` + `mask_radius=...`: ignore background pixels (zeros) in
  the correlation. **Always use a mask for real tomography data**.
- `apply_mean_filter_zyx=(3, 3, 3)`: optional 3D mean filter that
  suppresses noise and helps fine features correlate.
- `apply_affine_warp=True`: use an affine warp instead of pure
  translation. About 5× slower.


In [ ]:
# These are the registration parameters for this synthetic dataset.
# For real data, run `st.check_intersection(...)` first to decide on
# mask_radius, then tweak downscale / downscale_stages to fit the
# expected shift size.
st.erosion_mask_LC_xyz = (3, 3, 3)
st.add_value_for_mask = 0

st.compute_shift_in_layers(
    mask=True,
    mask_radius=8,           # px; smaller than for real data
    downscale=2,             # smaller because the shifts are small
    downscale_stages=2,
    downscale_LC=True,
    apply_mean_filter_zyx=(3, 3, 3),
    apply_affine_warp=False,
    keep_rigid_only=True,
    verbose=True,
)


### Inspect the registration result

The per-pair shifts and NCC scores are stored on the Stitcher. We can
print a one-line summary for every pair to sanity-check the result.


In [ ]:
for layer_idx, layer_shifts in enumerate(st.shifts):
    for (i, j), shift in layer_shifts.items():
        ncc = st.ncc_scores[layer_idx].get((i, j), float("nan"))
        print(f"  pair ({i}, {j})  shift(Z,Y,X)={tuple(shift.round(2).tolist())}  NCC={ncc:.3f}")


## 3. Accumulate displacements

Each neighbour pair has a **local** shift. The final stitched volume
needs a **global** shift for every sub-volume — relative to some
reference frame.

The Stitcher builds a **graph** of sub-volumes, takes a BFS from a
seed sub-volume, and **chains** the per-pair shifts along the path. If
two paths reach the same sub-volume, the NCC-weighted average is taken.
Low-NCC pairs can be pruned with `exclude_NCC=...`.

Math reference: [PUBLICATION].

### Parameters

- `exclude_NCC`: drop pairs with NCC below this threshold. A value of
  0.5–0.7 is typical; lower is more lenient.
- `weighted_avg`: average paths weighted by NCC. Strongly recommended.
- `affine_operator`: chain affine operators instead of pure translations.

Three methods run in sequence: `get_displacement_pyramid` builds the
graph, `accumulate_displacement` chains the shifts,
`compose_final_displacements` finalises the global warps.


In [ ]:
st.get_displacement_pyramid(check=False, starting_coord=(0, 0, 0))
st.accumulate_displacement(exclude_NCC=0.3, weighted_avg=True, verbose=False)
st.compose_final_displacements(verbose=False)

print("Global shifts (one per sub-volume):")
for i, shift in enumerate(st.global_shifts):
    print(f"  sub-volume {i}: shift(Z,Y,X) = {tuple(shift.round(2).tolist())}")


`check_accumulated_displacement` overlays the per-sub-volume global
shift on the canvas so you can visually confirm the graph is consistent
with the registration result.


In [ ]:
st.check_accumulated_displacement(verbose=False)


## 4 & 5. Equalisation and Blending

These two stages are usually called together:

- **Equalisation.** Adjacent scans can have different intensities —
  detector flux drifts, sample absorption changes. Equalisation fits a
  **linear map** in the overlap region of every pair, derived from the
  **joint histogram** of the two intensities. The result is a smooth
  intensity transition across the volume.
- **Blending.** Each sub-volume is warped onto the global canvas. A
  **distance map** weights every voxel (large in the centre, small at
  the border) so that no sub-volume dominates. Mask-aware interpolation
  keeps background (zero) pixels from bleeding into the foreground. The
  optional **GPU affine path** is 10–100× faster than CPU for large
  volumes.

Math reference: [PUBLICATION].

### Parameters

- `mask=True` + `mask_radius`: ignore background in the blend.
- `use_equalize=True`: run equalisation. The maps are cached in the
  Stitcher so re-runs of blending are free.
- `square_dist=True`: square the distance map (sharper falloff).
- `prop_x_y=(0, 1)`: propagate the distance along x or y only — useful
  for the rotation-stage case.


In [ ]:
st.affine_warp = False
st.prop_x_y = (0, 0)             # radial distance map
st.exclude_NCC = -200            # keep all pairs for blending

blended = st.stitch_volumes_blend_equalize(
    start_slice=st.img_depth // 2 - 2,
    end_slice=st.img_depth // 2 + 2,
    mask=True,
    mask_radius=8,
    use_equalize=True,
    use_existing_equalize=False,
    square_dist=False,
    crop_x=(0, 0),
    crop_y=(0, 0),
    show_progress_bar=False,
)
print("Blended slice-stack shape (Z, Y, X):", blended.shape)


In [ ]:
fig, axes = plt.subplots(1, blended.shape[0], figsize=(3 * blended.shape[0], 3))
for i in range(blended.shape[0]):
    axes[i].imshow(blended[i], cmap="gray")
    axes[i].set_title(f"blended Z={i}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()


## 6. Save and inspect

The blended stack above is a **preview** of the final result. To write
the full 3D volume, we:

1. Push the current parameters as the final ones (`push_stitch_parameters`).
2. Run `stitch_layers` to materialise every layer on the global canvas.
3. Read the result back from the `.h5` files.

The output is one `.h5` file per z-layer. Inside each file:

- `stitched_data/stitched_image` — the stitched volume chunk.
- `stitched_data/layer_coordinates` — the motor coordinates of the
  sub-volumes that contributed to this layer.
- `stitched_data/displacement_data` — the global shifts and operators,
  so you can re-run blending from the metadata without redoing the
  registration.


In [ ]:
st.push_stitch_parameters()
st.stitch_layers(
    chunk_size_series=40,
    chunk_size_parallel=10,
    n_cores=4,
    use_existing_equalize=True,
    path_save=os.path.join(WORK, "stitched"),
    check=False,
)


In [ ]:
out_dir = os.path.join(WORK, "stitched", "Stitched_layers")
out_files = sorted(f for f in os.listdir(out_dir) if f.endswith(".h5"))
print("Wrote", len(out_files), "layer file(s):", out_files)

with h5py.File(os.path.join(out_dir, out_files[0]), "r") as f:
    print("\nKeys in", out_files[0], ":", list(f.keys()))
    print("  stitched_data/", list(f["stitched_data"].keys()))
    vol = f["stitched_data/stitched_image"][:]
    print("  stitched_image shape (Z, Y, X):", vol.shape)

plt.figure(figsize=(5, 5))
plt.imshow(vol[vol.shape[0] // 2], cmap="gray")
plt.title("Final stitched volume (mid-Z)")
plt.axis("off")
plt.show()


## Wrap up

You have now run the **full six-stage pipeline** on a tiny synthetic
dataset, with a visualisation at every stage. To run on real data:

1. Replace `file_paths` with a list of your own `.h5` files.
2. Replace `centres_mm` with the actual motor positions (in mm).
3. Set `mm_per_vox` to the pixel size of your reconstructions.
4. Use `check_intersection` to pick a `mask_radius` that excludes the
   background.
5. Use `check_padding` to confirm the global canvas size makes sense
   for your data.
6. Re-tune `downscale`, `downscale_stages`, and `apply_affine_warp`
   based on the expected shift size and rotation.

If you have several z-layers, `get_layers_in_z` and `compute_shift_in_layers`
handle them automatically. For rotation-stage data, see
`notebooks/03_stitching_with_rotation.ipynb`.

**Mathematical details.** The ZNCC pixel search, the IC-GN
Lucas–Kanade optimiser, the mask-weighted correlation, and the
distance-map blending are described in [PUBLICATION].

**Troubleshooting.** Common errors (CUDA mismatch, low NCC, GPU OOM)
and how to recover are listed in [`docs/troubleshooting.md`]
(../docs/troubleshooting.md).
